# Traffic Demand — Feature Engineering
**Goal**: Turn raw columns into model-ready features using every insight from the EDA.  
**Sections**: Setup → Load → Impute → Time → Geohash → Road-type → Target → Validate → Save

## 0. Setup

In [ ]:
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
from pathlib import Path

DATA_DIR    = Path('../data')
OUT_DIR     = Path('../fe_outputs')
OUT_DIR.mkdir(parents=True, exist_ok=True)

TARGET       = 'demand'
CATEGORICALS = ['geohash', 'RoadType', 'LargeVehicles', 'Landmarks', 'Weather']
NUMERICS     = ['NumberofLanes', 'Temperature']

# TE smoothing strength (m-estimate). Higher k → more shrinkage toward global mean.
# k=10 is a safe default; tune during model CV if needed.
TE_K = 10

print('Libraries loaded. Data dir:', DATA_DIR.resolve())

> **What this cell does:** Mirrors the EDA setup cell — same tools, same paths.  
> We add one new constant, `TE_K=10`, which controls how strongly our geohash  
> target encoding is pulled toward the global average for locations with few rows.

## 1. Load Data

In [ ]:
train  = pd.read_csv(DATA_DIR / 'train.csv')
test   = pd.read_csv(DATA_DIR / 'test.csv')

# Tag each split so we can reunite them after joint transforms without mixing labels
train['_split'] = 'train'
test['_split']  = 'test'

# Concatenate for joint imputation and encoding — prevents train/test mismatch
full = pd.concat([train, test], axis=0, ignore_index=True)

print(f'train : {train.shape}')
print(f'test  : {test.shape}')
print(f'full  : {full.shape}')

> **Why concatenate?** Features like ordinal mappings and imputation values must be  
> consistent across both splits. We concatenate to apply transformations once, then  
> split back out at the end. The `_split` tag is our bookmark for separating them later.  
> **Note:** Target encoding (geohash mean demand) is computed on **train only** and  
> then mapped onto test — we never let test demand values leak into train.

## 2. Imputation

EDA found three columns with missing values — identical rates in train and test (MCAR). Simple fills are safe.

In [ ]:
# -------------------------------------------------------------------
# 2a.  Temperature — median fill
#      EDA: r=0.003 with demand (noise), 3.23% missing, mean ≈ 16.4 °C
#      We use the TRAIN median to avoid any leakage from test rows.
# -------------------------------------------------------------------
temp_median = train['Temperature'].median()
full['Temperature'] = full['Temperature'].fillna(temp_median)
print(f'Temperature nulls after fill : {full["Temperature"].isnull().sum()}  '
      f'(fill value = {temp_median:.2f} °C)')

# -------------------------------------------------------------------
# 2b.  RoadType — explicit "Missing" category
#      EDA: NaN rows behave like low-demand Residential (~0.098 mean).
#      Treating NaN as its own category lets the model learn this pattern
#      rather than forcing it into one of the three known road types.
# -------------------------------------------------------------------
full['RoadType'] = full['RoadType'].fillna('Missing')
print(f'RoadType  nulls after fill : {full["RoadType"].isnull().sum()}')

# -------------------------------------------------------------------
# 2c.  Weather — explicit "Missing" category
#      EDA: all 4 weather conditions produce identical demand (lifts ≈ 1.0).
#      Weather is noise, so any fill strategy is fine; "Missing" is safest.
# -------------------------------------------------------------------
full['Weather'] = full['Weather'].fillna('Missing')
print(f'Weather   nulls after fill : {full["Weather"].isnull().sum()}')

> **Imputation choices in plain English:**  
> - **Temperature**: filled with the training median (16.4 °C) because the correlation  
>   with demand is effectively zero — any sensible value is fine.  
> - **RoadType / Weather**: treated as a brand-new category "Missing" rather than  
>   forcing NaN into an existing bucket. This lets the model discover on its own  
>   how "Missing" rows behave, rather than pretending they are (say) Residential.

## 3. Time Features

The `timestamp` column is a string `'HH:MM'` on 15-minute intervals. We convert it to numeric and add cyclical encodings so the model understands that 23:45 and 00:00 are neighbours, not opposites.

In [ ]:
# -------------------------------------------------------------------
# 3a.  Parse timestamp → minutes_since_midnight
#      '2:15' → 2*60+15 = 135;  '13:45' → 13*60+45 = 825
#      Range: 0 (midnight) to 1425 (23:45).
# -------------------------------------------------------------------
def ts_to_minutes(ts_str):
    h, m = str(ts_str).split(':')
    return int(h) * 60 + int(m)

full['minutes'] = full['timestamp'].apply(ts_to_minutes)

# -------------------------------------------------------------------
# 3b.  Slot index  (0–95 within a full day)
#      Useful as a compact ordinal: slot 0 = 00:00, slot 95 = 23:45.
#      Some tree models prefer integer slot over raw minutes.
# -------------------------------------------------------------------
full['slot'] = full['minutes'] // 15

# -------------------------------------------------------------------
# 3c.  Cyclical (sin / cos) encoding
#      Maps the linear 0–1425 scale onto a circle so that the model
#      knows 23:45 and 00:00 are just 15 minutes apart.
#      angle = 2π × minutes / 1440  (1440 = total minutes in a day)
# -------------------------------------------------------------------
angle = 2 * np.pi * full['minutes'] / 1440
full['sin_slot'] = np.sin(angle)
full['cos_slot'] = np.cos(angle)

print(full[['timestamp', 'minutes', 'slot', 'sin_slot', 'cos_slot']].drop_duplicates()
        .sort_values('minutes').head(10).to_string(index=False))

> **Why sin/cos and not just minutes?**  
> A model that only sees `minutes` treats slot 0 (midnight) and slot 95 (23:45)  
> as 1425 units apart — almost as far apart as possible. But they are actually  
> 15 minutes apart on the daily clock. The sin/cos pair maps every slot onto a  
> unit circle: nearby times get nearby (sin, cos) values regardless of whether  
> they straddle midnight. This is the standard trick for any periodic feature  
> (hour of day, day of week, month of year).

## 4. Geohash Features

EDA rank: **#1 and #2 most important predictors.** A specific road location explains far more variance than any other single variable — the busiest geohash has mean demand 18× higher than the quietest. We create two geohash features:

1. **Smoothed target encoding** — the location's historical average demand, shrunk toward the global mean for locations with few rows  
2. **Per-(geohash, slot) mean** — the average demand at *this location at this specific time of day* on Day 48

In [ ]:
# -------------------------------------------------------------------
# 4a.  Smoothed target encoding  (m-estimate)
#
#      formula: te = (sum_demand + k × global_mean) / (n + k)
#      where k = TE_K (set to 10 in the Setup cell).
#
#      - For a geohash seen 105 times: k has ≈10% influence → mostly data-driven.
#      - For a geohash seen only 5 times: k has ≈67% influence → heavily shrunk.
#      - For the 10 unseen test geohashes: te falls back to the global mean itself.
#
#      IMPORTANT: compute statistics from TRAIN rows ONLY to avoid leakage.
#      We then map the encoded values onto both train and test rows.
# -------------------------------------------------------------------
train_mask = full['_split'] == 'train'

global_mean = full.loc[train_mask, TARGET].mean()
print(f'Global mean demand (train): {global_mean:.6f}')

gh_stats = (
    full.loc[train_mask]
    .groupby('geohash')[TARGET]
    .agg(gh_sum='sum', gh_n='count')
    .reset_index()
)
gh_stats['geohash_te'] = (gh_stats['gh_sum'] + TE_K * global_mean) / (gh_stats['gh_n'] + TE_K)

full = full.merge(gh_stats[['geohash', 'geohash_te']], on='geohash', how='left')

# Fallback for the 10 unseen test geohashes (0.84%) → global mean
full['geohash_te'] = full['geohash_te'].fillna(global_mean)

print(f'geohash_te range: [{full["geohash_te"].min():.4f}, {full["geohash_te"].max():.4f}]')
print(f'Null geohash_te  : {full["geohash_te"].isnull().sum()}  (should be 0)')

> **Smoothed target encoding in plain English:**  
> For each road location, we ask "what was the average demand here on Day 48?"  
> and store that as a number. Locations with many observations get the number  
> they earned; locations with only a handful of rows have their estimate gently  
> pulled toward the overall average — this prevents the model from over-trusting  
> a location it only saw once or twice.  
>  
> The 10 test geohashes that never appeared in training get the overall mean  
> as their fallback — the best guess we can make with no history.

In [ ]:
# -------------------------------------------------------------------
# 4b.  Per-(geohash, slot) mean demand from Day 48  ← strongest feature
#
#      This captures "how busy is *this exact road* at *this exact time of day*?"
#      It is the most information-rich, leak-free feature in the dataset because:
#        - Day 48 is fully observed in training (all 96 slots for all geohashes).
#        - The test set (Day 49, 02:15–13:45) shares the same time slots as Day 48.
#        - We never use any Day 49 demand values.
#
#      Edge cases handled:
#        - Geohash+slot not seen in train (very rare) → fall back to geohash_te
#        - Unseen test geohashes → fall back to global_mean
# -------------------------------------------------------------------
day48_mask = train_mask & (full['day'] == 48)

gh_slot_mean = (
    full.loc[day48_mask]
    .groupby(['geohash', 'slot'])[TARGET]
    .mean()
    .rename('geohash_slot_mean')
    .reset_index()
)

full = full.merge(gh_slot_mean, on=['geohash', 'slot'], how='left')

# Fill fallback: use geohash_te (already leak-free) for any gaps
full['geohash_slot_mean'] = full['geohash_slot_mean'].fillna(full['geohash_te'])

print(f'geohash_slot_mean range : [{full["geohash_slot_mean"].min():.4f}, {full["geohash_slot_mean"].max():.4f}]')
print(f'Null geohash_slot_mean  : {full["geohash_slot_mean"].isnull().sum()}  (should be 0)')
print(f'\nSample — top 5 busiest (geohash, slot) combos:')
print(gh_slot_mean.sort_values('geohash_slot_mean', ascending=False).head(5).to_string(index=False))

> **Why (geohash × slot) instead of just geohash?**  
> Even the busiest road has quiet periods (e.g., 3am) and peak periods (e.g., 9am).  
> Combining location and time of day gives us a much finer-grained estimate than  
> either alone. Think of it like a restaurant — knowing it's a popular restaurant  
> is useful, but knowing it's a popular restaurant *at 7pm on a weekday* is better.

## 5. Categorical Encoding

For the remaining categoricals (RoadType, LargeVehicles, Landmarks, Weather) we use **ordinal/label encoding** with a fixed map. LightGBM and XGBoost can treat these as `categorical_feature`, so a simple integer code is all that's needed — no one-hot expansion required.

In [ ]:
# -------------------------------------------------------------------
# 5a.  RoadType → ordinal (roughly ordered by mean demand from EDA)
#      Highway (0.611) > Street (0.273) > Missing (0.098) > Residential (0.057)
#      Encoding in descending demand order gives tree models a natural split axis.
# -------------------------------------------------------------------
roadtype_map = {'Highway': 3, 'Street': 2, 'Missing': 1, 'Residential': 0}
full['RoadType_enc'] = full['RoadType'].map(roadtype_map)

# Safety check: any values not in our map? (shouldn't happen — EDA confirmed 0 unseen)
unmapped = full['RoadType_enc'].isnull().sum()
if unmapped:
    print(f'WARNING: {unmapped} unmapped RoadType values — will be filled with -1')
    full['RoadType_enc'] = full['RoadType_enc'].fillna(-1)
print(f'RoadType_enc distribution:\n{full["RoadType_enc"].value_counts().sort_index().to_string()}')

In [ ]:
# -------------------------------------------------------------------
# 5b.  Binary categoricals — simple 0/1 encoding
#      LargeVehicles : Allowed → 1,  Not Allowed → 0
#      Landmarks     : Yes     → 1,  No          → 0
#      Both have 0 unseen values in test (EDA confirmed).
# -------------------------------------------------------------------
full['LargeVehicles_enc'] = (full['LargeVehicles'] == 'Allowed').astype(int)
full['Landmarks_enc']     = (full['Landmarks']     == 'Yes'    ).astype(int)

print(f'LargeVehicles_enc: {dict(full["LargeVehicles_enc"].value_counts())}')
print(f'Landmarks_enc    : {dict(full["Landmarks_enc"].value_counts())}')

In [ ]:
# -------------------------------------------------------------------
# 5c.  Weather → label encoding
#      EDA: all 4 conditions produce identical demand (lifts ≈ 1.0) — weather
#      is noise.  We still include it; the model's feature importance will
#      confirm near-zero weight.  A simple integer code is sufficient.
# -------------------------------------------------------------------
weather_map = {'Sunny': 0, 'Rainy': 1, 'Foggy': 2, 'Snowy': 3, 'Missing': 4}
full['Weather_enc'] = full['Weather'].map(weather_map).fillna(4).astype(int)

print(f'Weather_enc distribution:\n{full["Weather_enc"].value_counts().sort_index().to_string()}')

> **Why ordinal rather than one-hot for RoadType?**  
> One-hot creates multiple sparse columns that tree models split on one at a time  
> — no worse, but no better. An ordinal code in demand order is more compact and  
> gives the model a meaningful numeric axis to split on. LightGBM's  
> `categorical_feature` parameter handles these integer codes natively, so there  
> is no accuracy benefit to one-hot for tree-based models.

## 6. Interaction Features

EDA showed that RoadType and geohash are by far the two strongest signals. A few cheap interaction terms can help tree models discover combinations without growing too deep.

In [ ]:
# -------------------------------------------------------------------
# 6a.  is_highway / is_street  — binary flags for the two high-demand road types
#      EDA: Highway+Street together account for 96.7% of all top-decile demand.
#      A binary flag makes the split trivially learnable even in shallow trees.
# -------------------------------------------------------------------
full['is_highway'] = (full['RoadType'] == 'Highway').astype(int)
full['is_street']  = (full['RoadType'] == 'Street' ).astype(int)
full['is_highdemand_road'] = ((full['is_highway'] == 1) | (full['is_street'] == 1)).astype(int)

print(f'is_highway count          : {full["is_highway"].sum():,}')
print(f'is_street  count          : {full["is_street"].sum():,}')
print(f'is_highdemand_road count  : {full["is_highdemand_road"].sum():,}')

In [ ]:
# -------------------------------------------------------------------
# 6b.  RoadType × LargeVehicles interaction
#      EDA: Allowed roads are 1.8× busier, partly because they overlap with
#      highways.  An explicit product term helps the model isolate the
#      independent contribution of large vehicles (if any) beyond road type.
# -------------------------------------------------------------------
full['road_largeveh'] = full['RoadType_enc'] * full['LargeVehicles_enc']

print(f'road_largeveh sample:\n{full[["RoadType","LargeVehicles","road_largeveh"]].drop_duplicates().sort_values("road_largeveh").to_string(index=False)}')

In [ ]:
# -------------------------------------------------------------------
# 6c.  geohash_slot_mean × is_highdemand_road
#      Amplifies the location×time signal specifically for the road types
#      that matter most.  For Residential roads this is always zero, which
#      effectively silences the noisy near-zero tail when this feature is used.
# -------------------------------------------------------------------
full['gh_slot_x_road'] = full['geohash_slot_mean'] * full['is_highdemand_road']

print(f'gh_slot_x_road range: [{full["gh_slot_x_road"].min():.4f}, {full["gh_slot_x_road"].max():.4f}]')

## 7. Target Transform (Train Only)

EDA showed demand is severely right-skewed (skew = 3.73). We prepare both the raw and logit-transformed targets so we can compare them during model training.

In [ ]:
# -------------------------------------------------------------------
# 7a.  logit transform of demand
#      logit(p) = log(p / (1-p))
#      Clip to [ε, 1-ε] first to avoid ±inf at the exact 0 and 1 boundary.
#      At inference time: predicted demand = sigmoid(model_output) to get back
#      into [0, 1].
# -------------------------------------------------------------------
EPS = 1e-6

train_rows = full[full['_split'] == 'train'].copy()

demand_raw    = train_rows[TARGET]
demand_clipped = demand_raw.clip(EPS, 1 - EPS)
demand_logit   = np.log(demand_clipped / (1 - demand_clipped))
demand_log1p   = np.log1p(demand_raw)

print(f'Raw      demand — skew: {demand_raw.skew():.4f}')
print(f'log1p    demand — skew: {demand_log1p.skew():.4f}')
print(f'logit    demand — skew: {demand_logit.skew():.4f}')
print(f'\nlogit range: [{demand_logit.min():.2f}, {demand_logit.max():.2f}]')
print('\n>>> Logit is the preferred target transform (most symmetric, matches (0,1) support).')
print('    Compare holdout R² with raw vs logit during model training to confirm.')

> **Target transform summary:**  
> | Transform | Formula | Inverse | Skew |
> |-----------|---------|---------|------|
> | Raw | — | — | 3.73 |
> | log1p | log(1+x) | expm1(y) | reduced |
> | **logit** | log(x/(1-x)) | sigmoid(y) | near-zero |
>  
> We'll write `demand_logit` back into the training split as `target_logit`  
> and let the model CV decide which transform yields better R² on the holdout.

In [ ]:
# Write both transforms into the full dataframe (train rows only; test has no target)
full.loc[full['_split'] == 'train', 'target_logit'] = demand_logit.values
full.loc[full['_split'] == 'train', 'target_log1p'] = demand_log1p.values

print(f'target_logit nulls in train : {full.loc[full["_split"]=="train", "target_logit"].isnull().sum()}')
print(f'target_log1p nulls in train : {full.loc[full["_split"]=="train", "target_log1p"].isnull().sum()}')
print(f'target_logit nulls in test  : {full.loc[full["_split"]=="test",  "target_logit"].isnull().sum()}  (expected — test has no demand)')

## 8. Validation Split

EDA finding: this is a **forecasting problem** — test is strictly after train on the timeline. Random K-fold is invalid because it lets future slots leak into training folds. We must cut the training data along the time axis.

In [ ]:
# -------------------------------------------------------------------
# Validation design (from EDA section 4):
#   fit  : day 48, 00:00 – 21:45  (slots 0–87)
#   val  : day 48, 22:00 – 23:45  +  day 49, 00:00 – 02:00  (slots 88–95 + day-49 slots)
#
# This mirrors the exact temporal structure of train vs. test:
#   - Prediction target is always after the last fit row.
#   - The validation gap (8 tail slots on day 48 + 9 early day-49 slots)
#     approximates the ~12-hour forecasting horizon seen in test.
# -------------------------------------------------------------------
train_fe = full[full['_split'] == 'train'].copy()
test_fe  = full[full['_split'] == 'test' ].copy()

# Fit: day 48, slots 0–87 (00:00–21:45)
fit_mask = (train_fe['day'] == 48) & (train_fe['slot'] <= 87)
# Val: day 48 slots 88–95  +  all of day 49 in train
val_mask = ((train_fe['day'] == 48) & (train_fe['slot'] >= 88)) | (train_fe['day'] == 49)

fit_df = train_fe[fit_mask].copy()
val_df = train_fe[val_mask].copy()

print(f'Fit  rows : {len(fit_df):,}   slots {fit_df["slot"].min()}–{fit_df["slot"].max()} on day 48')
print(f'Val  rows : {len(val_df):,}   (day-48 tail + day-49 head)')
print(f'Test rows : {len(test_fe):,}   (day-49, slots {test_fe["slot"].min()}–{test_fe["slot"].max()})')
print(f'\nVal  demand stats: mean={val_df[TARGET].mean():.4f}  median={val_df[TARGET].median():.4f}')
print(f'Fit  demand stats: mean={fit_df[TARGET].mean():.4f}  median={fit_df[TARGET].median():.4f}')

> **Why this specific cut?**  
> We want our validation error to predict our test error as faithfully as possible.  
> Both test data and our validation set start right after the last training slot,  
> with no time gap or overlap. Using random KFold would let the model "see tomorrow"  
> during training — an unfair advantage that would make holdout scores look better  
> than they really are, and then disappoint on the real test submission.

## 9. Final Feature List & Sanity Check

In [ ]:
# -------------------------------------------------------------------
# Complete list of engineered features in priority order (from EDA)
# -------------------------------------------------------------------
FEATURE_COLS = [
    # ── Strongest: location × time interaction ──────────────────────
    'geohash_slot_mean',     # Per-(geohash, slot) mean demand from Day 48
    'geohash_te',            # Smoothed target encoding of geohash (overall location level)

    # ── Road type — near-perfect high-demand separator ───────────────
    'RoadType_enc',          # 0=Residential, 1=Missing, 2=Street, 3=Highway
    'is_highdemand_road',    # 1 if Highway or Street, else 0
    'is_highway',            # 1 if Highway
    'is_street',             # 1 if Street

    # ── Time of day ──────────────────────────────────────────────────
    'minutes',               # Raw minutes since midnight (0–1425)
    'slot',                  # 15-min slot index (0–95)
    'sin_slot',              # Cyclical sin encoding
    'cos_slot',              # Cyclical cos encoding

    # ── Road characteristics ──────────────────────────────────────────
    'NumberofLanes',         # 1–5; weak monotone signal, partial RoadType proxy
    'LargeVehicles_enc',     # 1=Allowed, 0=Not Allowed
    'Landmarks_enc',         # 1=Yes, 0=No; weak signal

    # ── Interactions ──────────────────────────────────────────────────
    'road_largeveh',         # RoadType_enc × LargeVehicles_enc
    'gh_slot_x_road',        # geohash_slot_mean × is_highdemand_road

    # ── Noise features (include; expect ~0 importance) ─────────────────
    'Temperature',           # Correlation with demand ≈ 0
    'Weather_enc',           # All conditions produce identical demand
]

# Verify no feature is null in either split
print(f'{"Feature":<25} {"Train nulls":>12} {"Test nulls":>12}')
print('─' * 52)
for col in FEATURE_COLS:
    tr_null = train_fe[col].isnull().sum()
    te_null = test_fe[col].isnull().sum()
    flag = '  ← CHECK' if (tr_null + te_null) > 0 else ''
    print(f'{col:<25} {tr_null:>12,} {te_null:>12,}{flag}')

In [ ]:
# Quick correlation check: new features vs demand (train only)
print(f'{"Feature":<25} {"Pearson r":>12}')
print('─' * 40)
for col in FEATURE_COLS:
    if col in train_fe.columns and train_fe[col].dtype in [float, int, 'float64', 'int64', 'int32']:
        r = train_fe[[col, TARGET]].dropna().corr().iloc[0, 1]
        print(f'{col:<25} {r:>+12.4f}')

> **What to look for in the correlation table:**  
> - `geohash_slot_mean` and `geohash_te` should show the highest positive correlations  
>   (location is the dominant signal from EDA).  
> - `RoadType_enc`, `is_highdemand_road`, `is_highway`, `is_street` should also be strongly positive.  
> - `Temperature` and `Weather_enc` should be near zero — consistent with EDA findings.  
> - If any feature shows unexpected values, re-check the engineering steps above.

## 10. Save Engineered Datasets

In [ ]:
# -------------------------------------------------------------------
# Save the four outputs we'll need for modelling:
#   train_fe.csv  — full training split with all engineered features
#   test_fe.csv   — test split with all engineered features (no demand column)
#   fit_fe.csv    — time-ordered fit portion for CV
#   val_fe.csv    — time-ordered validation portion for CV
# -------------------------------------------------------------------
save_cols = FEATURE_COLS + [TARGET, 'target_logit', 'target_log1p', 'day', 'geohash']

train_fe[save_cols].to_csv(OUT_DIR / 'train_fe.csv', index=False)
test_fe[FEATURE_COLS + ['day', 'geohash']].to_csv(OUT_DIR / 'test_fe.csv', index=False)
fit_df[save_cols].to_csv(OUT_DIR / 'fit_fe.csv',   index=False)
val_df[save_cols].to_csv(OUT_DIR / 'val_fe.csv',   index=False)

print('Saved:')
for name, df in [('train_fe.csv', train_fe), ('test_fe.csv', test_fe),
                 ('fit_fe.csv', fit_df),  ('val_fe.csv', val_df)]:
    print(f'  {name:<18} {df.shape[0]:>7,} rows × {len(FEATURE_COLS):>2} features')

In [ ]:
# -------------------------------------------------------------------
# Summary table: what each feature is and where it came from
# -------------------------------------------------------------------
summary = {
    'geohash_slot_mean'  : ('float', 'Day-48 mean demand per (geohash × slot)', 'EDA Rank 1'),
    'geohash_te'         : ('float', 'Smoothed target encoding of geohash (k=10)', 'EDA Rank 2'),
    'RoadType_enc'       : ('int',   'Ordinal map: Residential=0, Missing=1, Street=2, Highway=3', 'EDA Rank 3'),
    'is_highdemand_road' : ('int',   'Binary: 1 if Highway or Street', 'EDA Rank 3'),
    'is_highway'         : ('int',   'Binary: 1 if Highway', 'EDA Rank 3'),
    'is_street'          : ('int',   'Binary: 1 if Street', 'EDA Rank 3'),
    'minutes'            : ('int',   'Minutes since midnight (0–1425)', 'EDA Rank 4'),
    'slot'               : ('int',   '15-min slot index (0–95)', 'EDA Rank 4'),
    'sin_slot'           : ('float', 'Cyclical sin(2π × minutes / 1440)', 'EDA Rank 4'),
    'cos_slot'           : ('float', 'Cyclical cos(2π × minutes / 1440)', 'EDA Rank 4'),
    'NumberofLanes'      : ('int',   'Raw lane count 1–5; proxies RoadType', 'EDA Rank 6'),
    'LargeVehicles_enc'  : ('int',   'Binary: Allowed=1, Not Allowed=0', 'EDA Rank 7'),
    'Landmarks_enc'      : ('int',   'Binary: Yes=1, No=0', 'EDA Rank 8'),
    'road_largeveh'      : ('int',   'RoadType_enc × LargeVehicles_enc', 'Interaction'),
    'gh_slot_x_road'     : ('float', 'geohash_slot_mean × is_highdemand_road', 'Interaction'),
    'Temperature'        : ('float', 'Raw °C (median-imputed); EDA r≈0 with demand', 'Noise'),
    'Weather_enc'        : ('int',   'Label: Sunny=0, Rainy=1, Foggy=2, Snowy=3, Missing=4', 'Noise'),
}

print(f'{"Feature":<25} {"Type":<7} {"Origin":<12} Description')
print('─' * 100)
for feat, (dtype, desc, origin) in summary.items():
    print(f'{feat:<25} {dtype:<7} {origin:<12} {desc}')
print(f'\nTotal features: {len(summary)}')

> **Feature Engineering complete.** Outputs saved to `../fe_outputs/`.  
>  
> **Next step — Modelling (`02_model.ipynb`):**  
> 1. Load `fit_fe.csv` / `val_fe.csv`  
> 2. Train LightGBM on raw target and logit target, compare holdout R²  
> 3. Use `FEATURE_COLS` as the feature set  
> 4. Mark `RoadType_enc`, `LargeVehicles_enc`, `Landmarks_enc`, `Weather_enc` as `categorical_feature`  
> 5. Predict on `test_fe.csv`, apply sigmoid if logit model wins, clip to [0, 1]  
> 6. Submit `sample_submission.csv`-format file